In [1]:
with open("/kaggle/input/datasets/sara2graca/victorian-poetry/corpus.txt", "r", encoding="utf-8") as f:
    corpus = f.read()

In [2]:
chars = sorted(list(set(corpus)))
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

In [3]:
vocab_size = len(chars)
print(f"Vocabulary size: {vocab_size}")
print(f"Characters: {''.join(chars)}")

Vocabulary size: 201
Characters: 	
 !"#$%&'()*+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[]^_abcdefghijklmnopqrstuvwxyz{|}~£·ÀÁÄÆÇÈÉËÍÒÖÚàáâäæçèéêëíîïñòóôöùúûüÿĩŒœΑΕΠΣΤΦΧάέήίαβγδεζηθικλμνξοπρςστυφχωόύώἀἄἅἐἔἥὀὁὍὐὑὗὰὲὴὶὸῖῦῶ–—‘’“”•™


In [4]:
import torch
import torch.nn as nn

# the transformer block with a triangular causal mask, no flash attention

class TransformerBlockManual(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads,
                                          batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(),
            nn.Linear(d_ff, d_model))
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        normed = self.ln1(x)
        x = x + self.drop(
            self.attn(normed, normed, normed, attn_mask=mask)[0])
        x = x + self.drop(self.ff(self.ln2(x)))
        return x

In [5]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(),
            nn.Linear(d_ff, d_model))
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape

        normed = self.ln1(x)

        q = self.W_q(normed)
        k = self.W_k(normed)
        v = self.W_v(normed)

        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        attn = torch.nn.functional.scaled_dot_product_attention(
            q, k, v, dropout_p=self.drop.p, is_causal=True)

        attn = attn.transpose(1, 2).contiguous().view(B, T, C)
        attn = self.W_o(attn)

        x = x + self.drop(attn)
        x = x + self.drop(self.ff(self.ln2(x)))
        return x

In [6]:
def make_causal_mask(seq_len, device):
    # matrix where future positions are masked out
    mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1)
    mask = mask.masked_fill(mask == 1, float('-inf'))
    return mask

In [7]:
class GPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, context_len, dropout=0.1, use_flash=True):
        super().__init__()
        self.use_flash = use_flash
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(context_len, d_model)
        self.drop = nn.Dropout(dropout)

        block_class = TransformerBlock if use_flash else TransformerBlockManual
        self.blocks = nn.ModuleList([
            block_class(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.ln_final = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        seq_len = x.shape[1]
        positions = torch.arange(seq_len, device=x.device)
        x = self.drop(self.token_emb(x) + self.pos_emb(positions))

        if self.use_flash:
            for block in self.blocks:
                x = block(x)
        else:
            mask = make_causal_mask(seq_len, x.device)
            for block in self.blocks:
                x = block(x, mask=mask)

        x = self.ln_final(x)
        return self.head(x)

In [8]:
context_len = 256
d_model = 512
d_ff = 4 * d_model
nb_layers = 6
nb_heads = 8

In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = GPT(vocab_size, d_model, nb_heads, nb_layers, d_ff, context_len, dropout=0.1).to(device)

checkpoint = torch.load('/kaggle/input/models/sara2graca/512d/pytorch/default/1/checkpoint_step40000.pt', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

<All keys matched successfully>

In [10]:
def get_batch(data, batch_size, context_len, device):
    ix = torch.randint(len(data) - context_len, (batch_size,))
    x = torch.stack([data[i:i+context_len] for i in ix])
    y = torch.stack([data[i+1:i+context_len+1] for i in ix])
    return x.to(device), y.to(device)

In [11]:
from google.colab import files

def generate(model, prompt, max_new_chars, device, temperature=1.0):
    model.eval()

    x = torch.tensor([char_to_idx[c] for c in prompt], dtype=torch.long).unsqueeze(0).to(device)

    with torch.no_grad():
        for _ in range(max_new_chars):

            x_crop = x[:, -context_len:]
            logits = model(x_crop)

            logits = logits[:, -1, :] / temperature
            probs = torch.nn.functional.softmax(logits, dim=-1)
            next_char = torch.multinomial(probs, num_samples=1)
            x = torch.cat([x, next_char], dim=1)

    return ''.join([idx_to_char[i.item()] for i in x[0]])

In [12]:
# one stanza of a real poem
print(generate(model, prompt="EMILY JANE was a nursery maid,\n JAMES was a bold Life Guard,\n JOHN was a constable, poorly paid\n (And I am a doggerel bard).", max_new_chars=500, device=device))

EMILY JANE was a nursery maid,
 JAMES was a bold Life Guard,
 JOHN was a constable, poorly paid
 (And I am a doggerel bard).
  The foolish he must swallowed the shock
  He wept, but needfully answered him,
  But needfully that he had been dumb.
  So therefore I took my hearties would plain
  The stranger than a candlelish Bard.
  Forgiving his deed words is now behead,
  And for troublesome the dim deed;
  But then, in a word that he
  Was scantyding fire,
But the day seemed to marge it,
    He didn't change it allow!
Fun ye need for pupil they come
  But for puck and feed again,
For he thought the child was wonder,



In [13]:
# empty
print(generate(model, prompt=" ", max_new_chars=500, device=device))

               521

   TO A BILL OF NEW ENGLAND                                631

   THE NARCIES WOODCOCK                                              646

  THE SWORD'S SONG                                             646

  THE GHOSTS' HIGH NOON                               682

  MARY MAGDALEN                                              384

  THE RHYME (1682) A Piccadence of cards, unbrokered at seeing
      the editor's distaffardent shrink, inespicately was disgrace--
  And any thing th


In [14]:
# love theme
print(generate(model, prompt="Love, mariage and a girl", max_new_chars=500, device=device))

Love, mariage and a girl
      That poor little head, and was not Truth beside,
      Than when grandly beside the way near I make.

    Now _Dudla on the Mantelpiece_, where Nature looks at you go,
    And _I_ she changes past hange to see them and she.


'CHANGED, ALEXANDRE de VANTELLER


_Superchants_

  Not day, not to be wasted by a Peers of Senate--
    They are extremely rarestly eating a Colt's beaten.

                      _Sir Walter Scott._




          A WIFE


Sir Knyghte, and folks to ake yonder words:



In [15]:
# war theme
print(generate(model, prompt="War, peace, difficult army", max_new_chars=500, device=device))

War, peace, difficult army:
   We must wait; you tumble let them out!

   The Note is friendly over the link,--
      Everything, even never said,
   And, though it looked, and let Fate,
      My wonder saw a Commonweal miss,
   Nay, I was apise, or softer still note,
      And writhed it in his box,
   But he finds a fast--trusting fair or trust,
      Triust his beneath the dragon three;
   And, waking, he strays
      That stands are not wythered.

   “Gae sold me true up, gae to me,” good William,]
      All their st


In [47]:
print(generate(model, prompt="War, peace, difficult army", max_new_chars=500, device=device, temperature=0.3))

War, peace, difficult army,
      And make a prey to poke;
    And I was the man at my way
      That I can never did see.

    And I have a chuse, and I shall not forget,
      In my accustomed porters prepair,
    I have no courtesye of feeling in
      To fetch a wild frontest sorrow,
      And in this world we shall bring to ash
    The birds' done bust marshing sea,
      And white as sharp as the birds say.

    The wind it blew on the street,
      The blind as they blew burning back;
    The wind it blew out and 


In [16]:
# random letters
print(generate(model, prompt="A\n B\n C\n D\n E\n", max_new_chars=500, device=device))

A
 B
 C
 D
 E

  O' Dees!
  Fierce she became his life was in your hame,
  She came from her life to your shame,
  When they've got the league away;
  As fair as wife's years old
  Busily pauses me with Brochieftain:
  She am I with them rough--but didn't seem
  The sea-glint night silent on the floor,
  And I'll sing a single sparrow-white floor,
  For I'll flow--a sweet Marsh wind a sweet May, I ween,
  Wi' a bright and a bright air, and a fly
  I like my garden-plodded deep.

  She was a bright actor stood


In [17]:
# start of sentence
print(generate(model, prompt="In a ", max_new_chars=500, device=device))

In a virgin of thousand confidential Prince,
  But thousand pounds suggestead for railway to the Evil Lillia's to
       steel,
  And it's taken with sweets and perfect pillario, in a messages which
       day,
  On clouds with reverence and reign who is thankankably had done to wed
       first his beautiful brute to flitters.

  He reached the man replied, and gave the emotion
       scorched off in the amusement of his class,
  And he knows what then exposed knows what they're referring down,
    


In [18]:
# existing poem, low temperature
print(generate(model, prompt="EMILY JANE was a nursery maid,\n JAMES was a bold Life Guard,\n JOHN was a constable, poorly paid\n (And I am a doggerel bard).", max_new_chars=500, device=device, temperature=0.1))

EMILY JANE was a nursery maid,
 JAMES was a bold Life Guard,
 JOHN was a constable, poorly paid
 (And I am a doggerel bard).
  The College he poured the boys and a bit,
  And so I should have said a bit.

                         _James Russell Lowell._




                                                                                                                                                                                                                                                                                                                                                                              


In [19]:
# existing poem, high temperature
print(generate(model, prompt="EMILY JANE was a nursery maid,\n JAMES was a bold Life Guard,\n JOHN was a constable, poorly paid\n (And I am a doggerel bard).", max_new_chars=500, device=device, temperature=2))

EMILY JANE was a nursery maid,
 JAMES was a bold Life Guard,
 JOHN was a constable, poorly paid
 (And I am a doggerel bard).
  Foozd I pour and Phlepick's Teaser up in at Tea
  Made answer: "Misers," say,
  "|The Woman_ [treated Child!

 _Nothing to drIve._"

  But goblins late you, sir!
|Wins. Callab_ (You're di"away)--Spit, frang? 18.

 8--_Kitspunk. Lead
rpence: 18. Unne 94._)



'ROUND )η HEAVES A YARN.


Look out! O Hear, sax? All aréunda9;
  Of unything, smart itI, one.
If troth Saints are Scaector? æs! you're laughing,
 900--in Poem.

If you _dullect Dante_ commen, Madag's-Davourite gluest In!
And yit why "don


In [20]:
# low temperature
print(generate(model, prompt="In a ", device=device, max_new_chars=500, temperature=0.3))

In a sieve they went to sea:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            


In [21]:
# low temperature
print(generate(model, prompt="In a ", device=device, max_new_chars=500, temperature=0.3))

In a sieve they went to sea:
  But ah! I am with my wild soul and be!

                             _Anonymous_.




_A LOVE PLACE IN SAFE A LOVE AND A LOVE AND A LAURE_

  There was a lovely man in love beneath the shore,
      And a lovely man in love I love thee not;
      Ah, love me, love, is love, but love is loved before;
      Ah, love of love, love is never loved before;
      And bore be but be but one love beguiled by thee,
      And bore by the bore be browne bowhellows weth the stream,
 


In [22]:
# medium temperature
print(generate(model, prompt="In a ", device=device, max_new_chars=500, temperature=0.8))

In a gleaming land,
    And a glance of rosewood,
      And a sea before the dandeling of the dandeling,
    And a glimpse of great fold of figure that to death is good:
    But when she put out, she should not forget to fight,
      And for all his cheek or fight.

    Then we poor Friday at his boy,
      The youngest scraped on the shore;
    The fog came a court of bitter strong,
    A Bard of righteous caressed,
      Not wholly she sped;
    Then we poiled out the belt of Death,
      Then we p


In [23]:
# random letters, low temperature
print(generate(model, prompt="A\n B\n C\n D\n E\n", max_new_chars=500, device=device, temperature=0.3))

A
 B
 C
 D
 E
didn't he Ain't his tale,
For he said, and he should wed sae gent,
  And he should wed the poor and gray.

                           _James Jeffrey Roche._




                                                                                                                                                                                                                                                                                                                                                   


In [24]:
prompts = [
    "Was ever there found\n",
    "There was once a little man, and his rod and line he took,\n",
    "BY the side of a wood\n",
    "Oh, the days were ever shiny\n",
    "The day is done, and darkness\n",
    "Matilda Maud Mackenzie frankly hadn't any chin,\n",
    "Oh, what's the way to Arcady?\n",
    "Why do you wear your hair like a man,\n",
    "Of all the mismated pairs ever created\n",
    "Pour, varlet, pour the water,\n",
]

generated_poems = []
for prompt in prompts * 10:  # 10 times each = 100 poems
    poem = generate(model, prompt=prompt, max_new_chars=500, device=device, temperature=0.7)
    generated_poems.append(poem)

In [45]:
import re

def analyze_text(text):
    lines = [l for l in text.split('\n') if l.strip()]
    stanzas = [s.strip() for s in text.split('\n\n') if s.strip()]
    words = re.findall(r'[a-zA-Z]+', text.lower())
    words_orig = re.findall(r'[a-zA-Z]+', text)
    n = len(words)

    line_lengths = [len(l) for l in lines]
    line_word_counts = [len(re.findall(r'[a-zA-Z]+', l)) for l in lines]
    stanza_sizes = [len([l for l in s.split('\n') if l.strip()]) for s in stanzas]

    return {
        'avg_line_length_chars': sum(line_lengths) / len(line_lengths),
        'avg_line_length_words': sum(line_word_counts) / len(line_word_counts),
        'avg_stanza_size':       sum(stanza_sizes) / len(stanza_sizes),
        'vocab_richness':        len(set(words)) / n,
        'avg_word_length':       sum(len(w) for w in words) / n,
        'punct_density':         len(re.findall(r'[,.!?;:]', text)) / n,
        'lines_per_1000w':       len(lines) / n * 1000,
        'stanzas_per_1000w':     len(stanzas) / n * 1000,
        'numbers_per_1000w':     len(re.findall(r'\d+', text)) / n * 1000,
        'caps_per_1000w':        len([w for w in words_orig if w.isupper() and len(w) > 1]) / n * 1000,
    }

generated_text = '\n\n'.join(generated_poems)
cs = analyze_text(corpus)
gs = analyze_text(generated_text)

print(f"{'metric':35s} | {'corpus':>10} | {'generated':>10}")
print("-" * 60)
for metric in cs:
    print(f"{metric:35s} | {cs[metric]:>10.4f} | {gs[metric]:>10.4f}")

metric                              |     corpus |  generated
------------------------------------------------------------
avg_line_length_chars               |    39.2501 |    41.7816
avg_line_length_words               |     6.5940 |     7.0947
avg_stanza_size                     |     4.4098 |     3.2272
vocab_richness                      |     0.0456 |     0.2459
avg_word_length                     |     4.0597 |     3.8365
punct_density                       |     0.1683 |     0.1314
lines_per_1000w                     |   151.6530 |   140.9511
stanzas_per_1000w                   |    34.3899 |    43.6766
numbers_per_1000w                   |     6.5510 |     2.7369
caps_per_1000w                      |    22.4051 |    20.4128


In [28]:
for poem in generated_poems:
    print(f"{poem}\n")

Was ever there found
        Or ever the lover!




THE BISHOP OF RUM-TI-FOO AGAIN


I have felt the wonderness of the Akhoond,
  The wondering hours laid on the shore,
The work once more upon the stones the stones in poor,--
  Ah, fool, alas! fool, The foam and sigh!
      The silvery flagon trees that flow!

O Tom! ay, and win thy bright eyes
  With theel-cold crown in the chimbley blow,
And birds sang bear our love the soil in the skies,
  With the looks of the rose of the rose his breast-belled meally.

A little

There was once a little man, and his rod and line he took,
  And forty miles had got his waistcoat-handkerchief so inspired
  By round towards the air at his fellow-creatures of the beach;
  And Jack looked round for a space and patted he round at the health
       moon,
  And then he took his slack line of the hair of Jill looked through
        wheel of dry was the wheat this white closet?

  Is he talked the picture? the name of the thrill,
          O silly bringing be